In [23]:
import pandas as pd
import os

possible_paths = [
    "data/raw/dst-3.0_16_1_hh_database.csv",
    "./data/raw/dst-3.0_16_1_hh_database.csv",
    "../data/raw/dst-3.0_16_1_hh_database.csv",
]

for path in possible_paths:
    if os.path.exists(path):
        data_path = path
        break
else:
    raise FileNotFoundError("CSV not found")

hh = pd.read_csv(data_path, sep=";")

print("Raw dataset shape:", hh.shape)
hh.head()

Raw dataset shape: (44744, 12)


,"Пол, возраст",ЗП,Ищет работу на должность:,"Город, переезд, командировки",Занятость,График,Опыт работы,Последнее/нынешнее место работы,Последняя/нынешняя должность,Образование и ВУЗ,Обновление резюме,Авто
0,"Мужчина , 39 лет , родился 27 ноября 1979",29000 руб.,Системный администратор,"Советск (Калининградская область) , не готов к...","частичная занятость, проектная работа, полная ...","гибкий график, полный день, сменный график, ва...",Опыт работы 16 лет 10 месяцев Август 2010 — п...,"МАОУ ""СОШ № 1 г.Немана""",Системный администратор,Неоконченное высшее образование 2000 Балтийск...,16.04.2019 15:59,Имеется собственный автомобиль
1,"Мужчина , 60 лет , родился 20 марта 1959",40000 руб.,Технический писатель,"Королев , не готов к переезду , готов к редким...","частичная занятость, проектная работа, полная ...","гибкий график, полный день, сменный график, уд...",Опыт работы 19 лет 5 месяцев Январь 2000 — по...,Временный трудовой коллектив,"Менеджер проекта, Аналитик, Технический писатель",Высшее образование 1981 Военно-космическая ак...,12.04.2019 08:42,Не указано
2,"Женщина , 36 лет , родилась 12 августа 1982",20000 руб.,Оператор,"Тверь , не готова к переезду , не готова к ком...",полная занятость,полный день,Опыт работы 10 лет 3 месяца Октябрь 2004 — Де...,ПАО Сбербанк,Кассир-операционист,Среднее специальное образование 2002 Профессио...,16.04.2019 08:35,Не указано
3,"Мужчина , 38 лет , родился 25 июня 1980",100000 руб.,Веб-разработчик (HTML / CSS / JS / PHP / базы ...,"Саратов , не готов к переезду , готов к редким...","частичная занятость, проектная работа, полная ...","гибкий график, удаленная работа",Опыт работы 18 лет 9 месяцев Август 2017 — Ап...,OpenSoft,Инженер-программист,Высшее образование 2002 Саратовский государст...,08.04.2019 14:23,Не указано
4,"Женщина , 26 лет , родилась 3 марта 1993",140000 руб.,Региональный менеджер по продажам,"Москва , не готова к переезду , готова к коман...",полная занятость,полный день,Опыт работы 5 лет 7 месяцев Региональный мене...,Мармелад,Менеджер по продажам,Высшее образование 2015 Кгу Психологии и педаг...,22.04.2019 10:32,Не указано


In [24]:
import pandas as pd
import numpy as np

df = hh.copy()

# deleting full duplicates
df = df.drop_duplicates().reset_index(drop=True)

print("After removing duplicates:", df.shape)

After removing duplicates: (44591, 12)


In [25]:
TEXT_COLUMNS = [
    "Опыт работы",
    "Последняя/нынешняя должность",
    "Последнее/нынешнее место работы",
    "Образование и ВУЗ"
]

for col in TEXT_COLUMNS:
    df[col] = df[col].fillna("")

df["resume_text"] = (
    df["Опыт работы"] + " " +
    df["Последняя/нынешняя должность"] + " " +
    df["Последнее/нынешнее место работы"] + " " +
    df["Образование и ВУЗ"]
)

df["resume_text"] = df["resume_text"].str.replace(r"\s+", " ", regex=True).str.strip()

print("Empty resume_text:", (df["resume_text"] == "").sum())
print("Shape:", df.shape)

Empty resume_text: 0
Shape: (44591, 13)


In [26]:
df["label"] = (
    df["Ищет работу на должность:"]
    .astype(str)
    .str.strip()
)

# убираем пустые и "nan"
df = df[df["label"] != ""]
df = df[df["label"] != "nan"]

print("Total unique professions:", df["label"].nunique())
print("Shape:", df.shape)

Total unique professions: 14929
Shape: (44591, 14)


In [27]:
import re

def clean_label(text):
    text = str(text)

    # delete everything in parentheses
    text = re.sub(r"\(.*?\)", "", text)

    # leave only the part before the comma  
    text = text.split(",")[0]

    # leave only the part before the slash
    text = text.split("/")[0]

    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

df["label"] = df["label"].apply(clean_label)

print("Unique professions after structural cleaning:", df["label"].nunique())

Unique professions after structural cleaning: 8147


In [28]:
df["label"] = df["label"].str.lower().str.strip()

print("Unique professions after lower():", df["label"].nunique())

Unique professions after lower(): 7347


In [29]:
df["label"] = df["label"].str.replace(r"[^a-zа-я0-9\s\-]", "", regex=True)
df["label"] = df["label"].str.replace(r"\s+", " ", regex=True).str.strip()

print("Unique professions after symbol cleaning:", df["label"].nunique())

Unique professions after symbol cleaning: 7189


In [30]:
label_counts = df["label"].value_counts()

MIN_SAMPLES_PER_PROFESSION = 50
valid_labels = label_counts[label_counts >= MIN_SAMPLES_PER_PROFESSION].index

df = df[df["label"].isin(valid_labels)].copy()

print("Professions after filter:", len(valid_labels))
print("Dataset size after filter:", df.shape)

Professions after filter: 125
Dataset size after filter: (27550, 14)


In [31]:
# --- STEP: create supercategories (rule-based) ---

def map_to_supercategory(label):
    l = label.lower()

    # IT / Development
    if any(k in l for k in ["программист", "developer", "разработчик", "java", "python", "php", "frontend", "backend"]):
        return "IT_Development"

    # Data / Analytics
    if any(k in l for k in ["аналитик", "data", "bi", "ml", "machine learning"]):
        return "Data_Analytics"

    # DevOps / System / Infra
    if any(k in l for k in ["системный администратор", "devops", "инженер", "администратор"]):
        return "IT_Infrastructure"

    # Project / Product / Management
    if any(k in l for k in ["менеджер", "руководитель", "project", "product"]):
        return "Management"

    # Design
    if any(k in l for k in ["дизайнер", "ui", "ux"]):
        return "Design"

    # Marketing
    if any(k in l for k in ["маркетолог", "seo", "smm", "реклама"]):
        return "Marketing"

    # Support
    if any(k in l for k in ["поддержк", "support", "техническ специалист"]):
        return "Support"

    # Sales
    if any(k in l for k in ["продаж", "sales"]):
        return "Sales"

    # QA / Testing
    if any(k in l for k in ["qa", "тестировщик", "quality"]):
        return "QA"

    # Everything else
    return "Other"


df["supercategory"] = df["label"].apply(map_to_supercategory)

print("Supercategories:")
print(df["supercategory"].value_counts())
print("\nNumber of supercategories:", df["supercategory"].nunique())

Supercategories:
supercategory
IT_Infrastructure    8276
Management           7220
Other                5164
IT_Development       3197
Data_Analytics       1516
Support               916
Marketing             578
QA                    443
Design                240
Name: count, dtype: int64

Number of supercategories: 9


In [32]:
# --- inspect top labels for designing supercategories ---

label_counts = df["label"].value_counts()

print("Top 30 labels:")
print(label_counts.head(30))

print("\nLabels inside 'Other':")
print(
    df[df["supercategory"] == "Other"]["label"]
    .value_counts()
    .head(30)
)

Top 30 labels:
label
системный администратор               3941
инженер                               1308
аналитик                               915
руководитель проекта                   892
менеджер проектов                      841
руководитель проектов                  830
специалист технической поддержки       786
программист                            613
технический специалист                 569
менеджер проекта                       537
инженер-программист                    518
специалист по it                       508
менеджер                               505
менеджер интернет-магазина             469
специалист                             429
оператор                               417
сервисный инженер                      388
менеджер по продажам                   379
интернет-маркетолог                    368
it-специалист                          322
региональный менеджер по продажам      317
инженер технической поддержки          305
frontend-разработчик             

In [38]:
# --- STEP 3: refined supercategories (v3) ---

def map_to_supercategory_v4(label):
    l = label.lower()

    # --- Development ---
    if any(k in l for k in ["разработчик", "developer", "программист", "java", "python", "php", "frontend", "backend", "1с"]):
        return "IT_Development"

    # --- Data / Analytics ---
    if any(k in l for k in ["аналитик", "data", "bi", "ml", "machine learning"]):
        return "Data_Analytics"

    # --- QA ---
    if any(k in l for k in ["qa", "тестировщик", "quality"]):
        return "QA"

    # --- DevOps ---
    if any(k in l for k in ["системный администратор", "devops"]):
        return "IT_DevOps"

    # --- IT Support ---
    if any(k in l for k in ["поддержк", "support", "технической поддержки"]):
        return "IT_Support"

    # --- Security ---
    if any(k in l for k in ["информационной безопасности", "защите информации", "security"]):
        return "IT_Security"

    # --- Networking ---
    if any(k in l for k in ["связи", "телеком", "слаботочных"]):
        return "IT_Networking"

    # --- IT Generic ---
    if any(k in l for k in ["it-специалист", "специалист по it", "специалист it", "администратор сайта", "администратор баз данных", "it специалист"]):
        return "IT_Generic"

    # --- Hardware ---
    if any(k in l for k in ["монтажник", "техник", "ремонту компьютеров"]):
        return "IT_Hardware"

    # --- Technical Specialist ---
    if any(k in l for k in ["технический специалист"]):
        return "Technical_Specialist"

    # --- Generic Specialist ---
    if any(k in l for k in ["специалист", "ведущий специалист", "главный специалист"]):
        return "Generic_Specialist"

    # --- Project Management ---
    if any(k in l for k in ["руководитель проекта", "менеджер проекта", "project manager"]):
        return "Project_Management"

    # --- Coordination ---
    if any(k in l for k in ["координатор проекта", "администратор проектов"]):
        return "Project_Coordination"

    # --- Product ---
    if any(k in l for k in ["product"]):
        return "Product_Management"

    # --- Sales ---
    if any(k in l for k in ["продаж"]):
        return "Sales"

    # --- Marketing ---
    if any(k in l for k in ["маркетолог", "seo", "smm", "реклама"]):
        return "Marketing"

    # --- Content ---
    if any(k in l for k in ["контент", "копирайтер", "технический писатель"]):
        return "Content"

    # --- Design ---
    if any(k in l for k in ["дизайнер", "ui", "ux", "3d"]):
        return "Design"

    # --- Operations ---
    if any(k in l for k in ["оператор"]):
        return "Operations"

    # --- Engineering ---
    if any(k in l for k in ["инженер"]):
        return "Engineering"

    # --- Management ---
    if any(k in l for k in ["менеджер", "руководитель", "начальник", "директор"]):
        return "Management"

    return "Other"


df["supercategory_v3"] = df["label"].apply(map_to_supercategory_v3)

print(df["supercategory_v3"].value_counts())
print("\nNumber of categories:", df["supercategory_v3"].nunique())

supercategory_v3
Management              4932
IT_DevOps               4383
IT_Development          3197
Engineering             2533
Other                   2321
Project_Management      1623
Data_Analytics          1516
IT_Support              1221
IT_Generic              1137
Sales                    696
Operations               624
IT_Networking            622
Marketing                578
IT_Hardware              502
QA                       443
Content                  362
Design                   292
IT_Security              285
Project_Coordination     142
Product_Management       141
Name: count, dtype: int64

Number of categories: 20


In [39]:
df["supercategory_final"] = df["label"].apply(map_to_supercategory_v4)

print(df["supercategory_final"].value_counts())
print("\nNumber of categories:", df["supercategory_final"].nunique())

supercategory_final
Management              4932
IT_DevOps               4383
IT_Development          3197
Engineering             2533
Project_Management      1623
Data_Analytics          1516
Generic_Specialist      1356
IT_Support              1221
IT_Generic              1137
Sales                    696
Operations               624
IT_Networking            622
Technical_Specialist     569
Other                    550
IT_Hardware              502
QA                       443
Marketing                424
Content                  362
Design                   292
IT_Security              285
Project_Coordination     142
Product_Management       141
Name: count, dtype: int64

Number of categories: 22


In [40]:
# Inspect what is inside Other

other_counts = (
    df[df["supercategory_v2"] == "Other"]["label"]
    .value_counts()
    .head(40)
)

print(other_counts)

label
технический специалист                      569
специалист по it                            508
специалист                                  429
it-специалист                               322
помощник системного администратора          280
монтажник                                   225
ит-специалист                               191
администратор                               175
начинающий специалист                       135
мастер по ремонту компьютеров               107
ведущий специалист                          105
техник-электрик                             101
специалист по информационным технологиям     99
модератор                                    95
специалист it отдела                         92
специалист по работе с клиентами             88
главный специалист                           85
технический директор                         84
администратор сайта                          77
технический писатель                         76
администратор проектов            

In [11]:
label_counts = df["label"].value_counts()

print("Top 10 professions:")
print(label_counts.head(10))

print("\nBottom 10 professions (>=50):")
print(label_counts.tail(10))

Top 10 professions:
label
системный администратор             3941
инженер                             1308
аналитик                             915
руководитель проекта                 892
менеджер проектов                    841
руководитель проектов                830
специалист технической поддержки     786
программист                          613
технический специалист               569
менеджер проекта                     537
Name: count, dtype: int64

Bottom 10 professions (>=50):
label
руководитель интернет-проекта    57
руководитель группы              57
seo-оптимизатор                  56
аналитик бизнес-процессов        55
монтажник связи                  55
php-программист                  53
3d-моделлер                      52
руководитель ит                  52
программист java                 51
инженер асутп                    51
Name: count, dtype: int64


In [12]:
def normalize_simple_plural(text):
    words = text.split()
    normalized_words = []

    for w in words:
        if w.endswith("ов") and len(w) > 4:
            w = w[:-2]
        elif w.endswith("ы") and len(w) > 4:
            w = w[:-1]
        elif w.endswith("и") and len(w) > 4:
            w = w[:-1]
        normalized_words.append(w)

    return " ".join(normalized_words)

df["label"] = df["label"].apply(normalize_simple_plural)

label_counts = df["label"].value_counts()

print("Unique professions after simple morphology:",
      df["label"].nunique())

Unique professions after simple morphology: 125


In [ ]:
import re

def extract_gender_age(text):
    if pd.isna(text):
        return "Unknown", "Unknown"

    text = str(text).lower()

    # gender
    if "жен" in text:
        gender = "Female"
    elif "муж" in text:
        gender = "Male"
    else:
        gender = "Unknown"

    # age
    match = re.search(r"(\d{1,2})\s*лет", text)
    if match:
        age = int(match.group(1))
        if age < 18:
            age_group = "<18"
        elif age <= 21:
            age_group = "18–21"
        elif age <= 25:
            age_group = "22–25"
        elif age <= 35:
            age_group = "26–35"
        elif age <= 50:
            age_group = "36–50"
        else:
            age_group = "50+"
    else:
        age_group = "Unknown"

    return gender, age_group


df[["gender", "age_group"]] = (
    df["Пол, возраст"]
    .apply(extract_gender_age)
    .apply(pd.Series)
)

In [16]:
print(df["gender"].value_counts())
print(df["age_group"].value_counts())

gender
Male      22291
Female     5259
Name: count, dtype: int64
age_group
Unknown    11229
26–35       9643
36–50       4491
22–25       1468
18–21        391
50+          314
<18           14
Name: count, dtype: int64


In [17]:
df["city"] = (
    df["Город, переезд, командировки"]
    .astype(str)
    .str.split(",")
    .str[0]
    .str.strip()
)

print("Top 10 cities:")
print(df["city"].value_counts().head(10))

print("\nUnique cities:", df["city"].nunique())

Top 10 cities:
city
Москва             9333
Санкт-Петербург    2951
Краснодар           704
Новосибирск         595
Казань              579
Екатеринбург        487
Самара              479
Нижний Новгород     389
Ростов-на-Дону      384
Уфа                 375
Name: count, dtype: int64

Unique cities: 842


In [18]:
MIN_CITY_SUPPORT = 100

city_counts = df["city"].value_counts()
major_cities = city_counts[city_counts >= MIN_CITY_SUPPORT].index

df["city_group"] = df["city"].where(
    df["city"].isin(major_cities),
    "Other"
)

print("City groups:")
print(df["city_group"].value_counts())
print("\nNumber of city groups:", df["city_group"].nunique())

City groups:
city_group
Москва              9333
Other               6276
Санкт-Петербург     2951
Краснодар            704
Новосибирск          595
Казань               579
Екатеринбург         487
Самара               479
Нижний Новгород      389
Ростов-на-Дону       384
Уфа                  375
Воронеж              350
Алматы               334
Пермь                303
Красноярск           281
Тюмень               251
Минск                210
Омск                 203
Челябинск            198
Ярославль            194
Томск                181
Волгоград            181
Саратов              169
Иркутск              156
Владивосток          141
Барнаул              137
Калининград          134
Хабаровск            131
Тольятти             127
Ижевск               127
Оренбург             120
Липецк               118
Тверь                111
Ульяновск            111
Пенза                106
Ставрополь           105
Калуга               105
Сочи                 105
Кемерово             104
С

In [19]:
df_final = df[[
    "resume_text",
    "label",
    "city_group",
    "gender",
    "age_group"
]].copy()

print("Final dataset shape:", df_final.shape)
print("Unique professions:", df_final["label"].nunique())

Final dataset shape: (27550, 5)
Unique professions: 125


In [20]:
from sklearn.model_selection import train_test_split

X = df_final["resume_text"]
y = df_final["label"]

X_temp, X_test, y_temp, y_test, df_temp, df_test = train_test_split(
    X, y, df_final,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val, df_train, df_val = train_test_split(
    X_temp, y_temp, df_temp,
    test_size=0.25,  # 0.25 * 0.8 = 0.2
    random_state=42,
    stratify=y_temp
)

print("Train:", len(df_train))
print("Val:", len(df_val))
print("Test:", len(df_test))

Train: 16530
Val: 5510
Test: 5510


In [22]:
import os

BASE_DIR = ".."  # leaving notebooks/
PROCESSED_DIR = os.path.join(BASE_DIR, "data", "processed")

os.makedirs(PROCESSED_DIR, exist_ok=True)

df_train.to_csv(os.path.join(PROCESSED_DIR, "train.csv"), index=False)
df_val.to_csv(os.path.join(PROCESSED_DIR, "val.csv"), index=False)
df_test.to_csv(os.path.join(PROCESSED_DIR, "test.csv"), index=False)

print("Datasets saved to:", PROCESSED_DIR)

Datasets saved to: ../data/processed
